In [ ]:
import os
import pandas as pd
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

c:\Users\marin\Documents\TUC\7_datakvalitet\API\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Load the CSV file and create documents
csv_path = "airbnb_cleaned.csv"

loader = CSVLoader(file_path=csv_path, encoding="utf-8")
docs = loader.load()

print(len(docs))
print(docs[0])

90567
page_content='id: 13913
name: Holiday London DB Room Let-on going
host_id: 54730
neighbourhood: Islington
room_type: Private room
price: 70.0
minimum_nights: 1
number_of_reviews: 55
calculated_host_listings_count: 2
availability_365: 331
number_of_reviews_ltm: 10
license: Unknown
city: London' metadata={'source': 'airbnb_cleaned.csv', 'row': 0}


In [ ]:
# text_splitters is not needed here since it's csv data

In [ ]:
# Create embeddings and store in Chroma vector database, we use HuggingFaceEmbeddings instead of GoogleGenerativeAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5579.77it/s]


In [ ]:
# Create a Chroma vector store
vector_store = Chroma(
    collection_name="airbnb",
    embedding_function=embeddings,
    persist_directory="./chroma_airbnb_db",
)

In [ ]:
# Add documents to the vector store in batches and print progress 
batch_size = 1000

for start in range(0, len(docs), batch_size):
    end = start + batch_size
    batch = docs[start:end]

    vector_store.add_documents(batch)

    print(f"Added documents {start} to {start + len(batch) - 1}")

Added documents 0 to 999
Added documents 1000 to 1999
Added documents 2000 to 2999
Added documents 3000 to 3999
Added documents 4000 to 4999
Added documents 5000 to 5999
Added documents 6000 to 6999
Added documents 7000 to 7999
Added documents 8000 to 8999
Added documents 9000 to 9999
Added documents 10000 to 10999
Added documents 11000 to 11999
Added documents 12000 to 12999
Added documents 13000 to 13999
Added documents 14000 to 14999
Added documents 15000 to 15999
Added documents 16000 to 16999
Added documents 17000 to 17999
Added documents 18000 to 18999
Added documents 19000 to 19999
Added documents 20000 to 20999
Added documents 21000 to 21999
Added documents 22000 to 22999
Added documents 23000 to 23999
Added documents 24000 to 24999
Added documents 25000 to 25999
Added documents 26000 to 26999
Added documents 27000 to 27999
Added documents 28000 to 28999
Added documents 29000 to 29999
Added documents 30000 to 30999
Added documents 31000 to 31999
Added documents 32000 to 32999
A

In [ ]:
# Test the chatbot with a sample question
question = "How much does airbnb cost in London?"

results = vector_store.similarity_search(question, k = 3)
for res in results:
    print(f"{res.page_content}\n")

id: 13612478
name: Extra Large Bedroom - 15 minutes into London.
host_id: 78788474
neighbourhood: Haringey
room_type: Private room
price: 52.0
minimum_nights: 2
number_of_reviews: 3
calculated_host_listings_count: 1
availability_365: 225
number_of_reviews_ltm: 3
license: Unknown
city: London

id: 26872240
name: Stylish 2 bedroom flat 30 mins to London*parking*
host_id: 4758004
neighbourhood: Sutton
room_type: Entire home/apt
price: 69.0
minimum_nights: 2
number_of_reviews: 85
calculated_host_listings_count: 1
availability_365: 251
number_of_reviews_ltm: 12
license: Unknown
city: London

id: 8768842
name: Downstairs Ensuite double bedroom to rent £40 pn
host_id: 45974964
neighbourhood: Bromley
room_type: Private room
price: 42.0
minimum_nights: 2
number_of_reviews: 32
calculated_host_listings_count: 1
availability_365: 357
number_of_reviews_ltm: 1
license: Unknown
city: London



In [7]:
import pandas as pd
df = pd.read_csv(csv_path)

In [8]:
# Number of rows in the dataset
print(len(df))
# Number of unique cities in the dataset
print(df['city'].nunique())
# Average price in each city in the dataset
print(df[df['city'] == 'London']['price'].mean())  
print(df[df['city'] == 'Rome']['price'].mean())  
print(df[df['city'] == 'Barcelona']['price'].mean())  
print(df[df['city'] == 'Amsterdam']['price'].mean())  

90567
4
170.84698298415623
167.6930775357551
197.4393360270831
257.49747580984433


In [ ]:
# Create a summary of the dataset so that the chatbot can use it to answer questions about the dataset
from langchain_core.documents import Document

summary_text = f"""
Dataset summary:

Number of listings: {90567}
Number of cities: {4}
Average price of London: {170.8}
Average price of Rome: {167.70}
Average price of Barcelona: {197.4}
Average price of Amsterdam: {257.50}
Currency: USD
Dataset type: Airbnb listings
"""

print(summary_text)


Dataset summary:

Number of listings: 90567
Number of cities: 4
Average price of London: 170.8
Average price of Rome: 167.7
Average price of Barcelona: 197.4
Average price of Amsterdam: 257.5
Currency: USD
Dataset type: Airbnb listings



In [ ]:
# Create a new document from the summary text and add metadata to it
summary_doc = Document(
    page_content=summary_text,
    metadata={"type": "dataset_summary"}
)

In [ ]:
# Add the document to the vector store 
vector_store.add_documents(documents=[summary_doc])

['91300f94-5682-455f-94a2-2ee43261a003']

In [ ]:

from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

In [ ]:
# Initialize the Google Generative AI model and retriever and create a prompt template for answering questions about the dataset
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.runnables import RunnablePassthrough 
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

model = ChatGoogleGenerativeAI(model="gemini-3-flash-preview", google_api_key=api_key)
retriever = vector_store.as_retriever()

template = """
Answer the question in the style of a kind European tour guide.

Instructions: 
Answer in a unique and detailed way.
If you do not find the answer to the question, please answer that you don't know the answer and show examples of questions you can answer.

Context:
{context}

Question:
{question}
"""

prompt= PromptTemplate.from_template(template)

In [ ]:
# Create a chain that combines the retriever, prompt, model, and output parser 
parser = StrOutputParser()

chain = (
    {
        "context": retriever, 
        "question":  RunnablePassthrough()
    }
    | prompt
    | model
    | parser
)

In [ ]:
# "Can you recommend some Airbnb accommodations in Rome with their price range? It would be great if they have many reviews."
question = input("Write your question here: ")

In [16]:
answer = chain.invoke(question)
print(answer)

*Buongiorno!* Welcome to the Eternal City! It is such a pleasure to help a fellow traveler find a beautiful place to rest after a long day of wandering through our sun-drenched piazzas and ancient ruins. 

Rome is a city of layers, and where you choose to stay is the first step in your Roman adventure. Looking at my records, I have found a few lovely options for you, ranging from cozy hideaways to spots with breathtaking views.

Here are a few recommendations based on what you’re looking for:

*   **For the Best View (and the most feedback):** 
    If you wish to wake up and see the majesty of the city, I highly recommend the **"Apartment with view of Rome"** located in the heart of the *I Centro Storico*. It has been quite popular with 34 reviews from previous guests. It is a bit of a splurge at **€264.00 per night**, but staying in the historic center with such a view is truly a unique experience!

*   **For Style and Value:** 
    Just a short stroll away, also in the *I Centro Stor

In [17]:
# "How many airbnbs are pet friendly in London?", to see if it halucinates or not.
question2 = input("Write your question here: ")

In [18]:
answer2 = chain.invoke(question2)
print(answer2)

Greetings! It is such a pleasure to meet you. I see you are planning a grand adventure to our historic city of London and—even better—you are looking to bring a furry companion along! London is a marvelous place for a stroll with a pet, especially with all our beautiful green heaths and riverside paths.

From the specific collection of properties I have in my guidebook today, I have found **three** wonderful pet-friendly options for you to consider:

1.  **In Greenwich:** There is a "cozy-cool" urban apartment priced at a very reasonable £110 per night. It’s a perfect spot for those who enjoy maritime history and the lovely views of the Thames.
2.  **In Hackney:** For those planning a more leisurely visit, there is an entire home available for £160 per night. They even offer a special discount for long stays during the beautiful months of April and May!
3.  **In Hounslow:** If you are traveling by car, there is a delightful studio for £170 per night that offers the rare and precious gi